In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import copy

# 1. Device Selection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Data Reading and Time Coding (Fragmentation Protected)
print("Data is loaded and Time Coding is applied...")
df_raw = pd.read_csv('time_series_60min_singleindex.csv', parse_dates=['utc_timestamp'], index_col='utc_timestamp')
base_features = ['AT_solar_generation_actual', 'AT_wind_onshore_generation_actual', 'AT_load_actual_entsoe_transparency']
data = df_raw[base_features].copy().dropna()

time_features = pd.DataFrame(index=data.index)
time_features['hour'] = time_features.index.hour
time_features['month'] = time_features.index.month
time_features['hour_sin'] = np.sin(2 * np.pi * time_features['hour'] / 24.0)
time_features['hour_cos'] = np.cos(2 * np.pi * time_features['hour'] / 24.0)
time_features['month_sin'] = np.sin(2 * np.pi * time_features['month'] / 12.0)
time_features['month_cos'] = np.cos(2 * np.pi * time_features['month'] / 12.0)

data = pd.concat([data, time_features[['hour_sin', 'hour_cos', 'month_sin', 'month_cos']]], axis=1)

# 3. Scaling
scaler = MinMaxScaler(feature_range=(-1, 1))
data_scaled = scaler.fit_transform(data.values)

# 4. Sliding Window
def create_daily_sequences(data_array, lookback=48, horizon=24):
    X, y = [], []
    for i in range(len(data_array) - lookback - horizon + 1):
        X.append(data_array[i : (i + lookback), :])
        y.append(data_array[(i + lookback) : (i + lookback + horizon), :].flatten()) 
    return np.array(X), np.array(y)

X, y = create_daily_sequences(data_scaled, 48, 24)

# ===========================================
# 5. ARCHITECTURAL DEFINITIONS
# ===========================================
INPUT_SIZE = 7
OUTPUT_SIZE = 7 * 24 

class GRU_Model(nn.Module):
    def __init__(self):
        super(GRU_Model, self).__init__()
        self.gru = nn.GRU(input_size=INPUT_SIZE, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(64, OUTPUT_SIZE)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

class LSTM_Model(nn.Module):
    def __init__(self):
        super(LSTM_Model, self).__init__()
        self.lstm = nn.LSTM(input_size=INPUT_SIZE, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(64, OUTPUT_SIZE)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class BiLSTM_Model(nn.Module):
    def __init__(self):
        super(BiLSTM_Model, self).__init__()
        self.lstm = nn.LSTM(input_size=INPUT_SIZE, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
        self.fc = nn.Linear(128, OUTPUT_SIZE) 
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

# ===========================================
# 6. NET LOAD TRAINING CYCLE WITH CROSS VERIFICATION (CV)
# ===========================================
def run_cv_net_load(ModelClass, model_name, lr=0.001):
    n_splits = 3
    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    net_load_wape_scores = []
    ramp_mae_scores = []
    
    print(f"\n--- {model_name} NET YÜK VE ESNEKLİK ÇAPRAZ DOĞRULAMASI BAŞLADI ---")
    
    fold = 1
    for train_index, test_index in tscv.split(X):
        fold_train_size = int(len(train_index) * 0.85)
        real_train_idx = train_index[:fold_train_size]
        val_idx = train_index[fold_train_size:]
        
        X_train, y_train = torch.from_numpy(X[real_train_idx]).float(), torch.from_numpy(y[real_train_idx]).float()
        X_val, y_val = torch.from_numpy(X[val_idx]).float(), torch.from_numpy(y[val_idx]).float()
        X_test, y_test = torch.from_numpy(X[test_index]).float(), torch.from_numpy(y[test_index]).float()

        train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=False)
        val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64, shuffle=False)
        
        # We start the model from scratch at each fold.
        model = ModelClass().to(device)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        
        best_val_loss = float('inf')
        best_weights = copy.deepcopy(model.state_dict())
        patience = 5
        epochs_no_improve = 0
        
        # Education
        for epoch in range(25):
            model.train()
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()
                loss = criterion(model(X_batch), y_batch)
                loss.backward()
                optimizer.step()
                
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                    val_loss += criterion(model(X_batch), y_batch).item()
            val_loss /= len(val_loader)
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_weights = copy.deepcopy(model.state_dict())
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve == patience:
                    break
                    
        # Testing Phase
        model.load_state_dict(best_weights)
        model.eval()
        with torch.no_grad():
            preds = model(X_test.to(device)).cpu().numpy()
            
        # Inverse Scaling
        preds_mw = scaler.inverse_transform(preds.reshape(-1, INPUT_SIZE)).reshape(preds.shape)
        y_test_mw = scaler.inverse_transform(y_test.numpy().reshape(-1, INPUT_SIZE)).reshape(y_test.shape)

        actual_solar = y_test_mw[:, 0::INPUT_SIZE]
        actual_wind  = y_test_mw[:, 1::INPUT_SIZE]
        actual_load  = y_test_mw[:, 2::INPUT_SIZE]
        
        pred_solar = preds_mw[:, 0::INPUT_SIZE]
        pred_wind  = preds_mw[:, 1::INPUT_SIZE]
        pred_load  = preds_mw[:, 2::INPUT_SIZE]

        # 1. NET LOAD CALCULATION
        actual_net_load = actual_load - (actual_solar + actual_wind)
        pred_net_load = pred_load - (pred_solar + pred_wind)
        
        net_load_mae = mean_absolute_error(actual_net_load, pred_net_load)
        net_load_wape = (net_load_mae / np.mean(np.abs(actual_net_load))) * 100

        # 2. FLEXIBILITY (RAMPING) CALCULATION
        actual_ramp = np.abs(np.diff(actual_net_load, axis=1))
        pred_ramp = np.abs(np.diff(pred_net_load, axis=1))
        ramp_mae = mean_absolute_error(actual_ramp, pred_ramp)
        
        print(f"  [Fold {fold}] Net Yük WAPE: %{net_load_wape:.2f} | Ramping MAE: {ramp_mae:.2f} MW")
        
        net_load_wape_scores.append(net_load_wape)
        ramp_mae_scores.append(ramp_mae)
        fold += 1

    print(f"[{model_name}] NİHAİ ORTALAMA -> Net Yük WAPE: %{np.mean(net_load_wape_scores):.2f} | Ramping MAE: {np.mean(ramp_mae_scores):.2f} MW")
    print("="*60)

# ===========================================
# 7. OPERATION
# ===========================================
# We send the classes of the models to the function as references
run_cv_net_load(GRU_Model, "GRU")
run_cv_net_load(LSTM_Model, "LSTM")
run_cv_net_load(BiLSTM_Model, "Bi-LSTM")